# 📝 Multithreading & Multiprocessing
### Exercises & Solutions — 25 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- Basic threading: Thread, join, args (1-3)
- Synchronization: Lock, RLock, Event, Condition, Semaphore (4-9)
- ThreadPoolExecutor patterns (10-13)
- Multiprocessing basics: Process, Pool (14-17)
- ProcessPoolExecutor & IPC: Queue, Value, Manager (18-21)
- Real-world hybrid patterns & pitfalls (22-25)


---


### 1. Basic Thread Creation and join()

Create 3 threads each printing a message after a delay, and use `.join()` to wait for ALL of them before continuing.

In [ ]:
import threading, time

def worker(name, delay):
    time.sleep(delay)
    print(f"{name} finished after {delay}s")

threads = [threading.Thread(target=worker, args=(f"Worker-{i}", 0.05*i)) for i in range(1, 4)]
start = time.perf_counter()
for t in threads: t.start()
for t in threads: t.join()
print(f"All threads joined after {time.perf_counter()-start:.2f}s")

### 2. Thread Returning a Value via a Mutable Container

Since `Thread.run()` can't return a value directly, capture a thread's result using a shared list/dict passed by reference.

In [ ]:
import threading

def compute_square(n, results, index):
    results[index] = n * n

results = [None] * 5
threads = [threading.Thread(target=compute_square, args=(i, results, i)) for i in range(5)]
for t in threads: t.start()
for t in threads: t.join()
print(results)

### 3. Daemon Threads vs Normal Threads

Demonstrate the difference between a daemon thread (dies when main program exits) and a normal thread (keeps the program alive) conceptually with `.daemon` flag.

In [ ]:
import threading, time

def background_task():
    time.sleep(0.1)
    print("Background task completed")

t = threading.Thread(target=background_task, daemon=True)
print("Daemon flag set:", t.daemon)
t.start()
t.join()   # we still wait here for the demo, but in real programs daemon threads
print("Main thread continues; if main exited without join(), daemon thread would be killed")

### 4. Lock to Prevent Race Conditions

Run 8 threads incrementing a shared counter 1000 times each, WITHOUT a lock (showing wrong results), then WITH a lock (correct results).

In [ ]:
import threading

def unsafe_increment(counter, times):
    for _ in range(times):
        counter[0] += 1

counter = [0]
threads = [threading.Thread(target=unsafe_increment, args=(counter, 1000)) for _ in range(8)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Unsafe (expected 8000): {counter[0]}")

lock = threading.Lock()
def safe_increment(counter, times):
    for _ in range(times):
        with lock:
            counter[0] += 1

counter = [0]
threads = [threading.Thread(target=safe_increment, args=(counter, 1000)) for _ in range(8)]
for t in threads: t.start()
for t in threads: t.join()
print(f"Safe (expected 8000): {counter[0]}")

### 5. RLock for Reentrant Locking

Show why a regular `Lock` would deadlock if a thread tries to acquire it twice (recursively), and how `RLock` solves this.

In [ ]:
import threading

rlock = threading.RLock()

def outer():
    with rlock:
        print("Acquired outer")
        inner()

def inner():
    with rlock:                  # SAME thread re-acquiring - fine with RLock, would deadlock with Lock
        print("Acquired inner (same thread, reentrant)")

t = threading.Thread(target=outer)
t.start()
t.join()
print("Completed without deadlock thanks to RLock")

### 6. Condition for Wait/Notify Coordination

Use `threading.Condition` so a 'consumer' thread waits until a 'producer' thread signals that data is ready.

In [ ]:
import threading, time

condition = threading.Condition()
data = {"ready": False, "value": None}

def producer():
    time.sleep(0.1)
    with condition:
        data["value"] = 42
        data["ready"] = True
        print("Producer: data ready, notifying")
        condition.notify()

def consumer():
    with condition:
        print("Consumer: waiting for data...")
        condition.wait_for(lambda: data["ready"])
        print(f"Consumer: got value {data['value']}")

t1 = threading.Thread(target=consumer)
t2 = threading.Thread(target=producer)
t1.start(); t2.start()
t1.join(); t2.join()

### 7. Event for One-to-Many Signaling

Use `threading.Event` to release MULTIPLE waiting threads simultaneously once a single event is set (unlike Condition's single notify).

In [ ]:
import threading, time

event = threading.Event()

def waiter(name):
    print(f"{name}: waiting for go signal...")
    event.wait()
    print(f"{name}: GO!")

threads = [threading.Thread(target=waiter, args=(f"Runner-{i}",)) for i in range(3)]
for t in threads: t.start()

time.sleep(0.1)
print("Starter: firing the event for ALL waiters at once")
event.set()
for t in threads: t.join()

### 8. Semaphore Limiting Concurrent Thread Access

Use a `threading.Semaphore(2)` to ensure at most 2 threads access a 'limited resource' simultaneously out of 6 total threads.

In [ ]:
import threading, time

sem = threading.Semaphore(2)
active = []
lock = threading.Lock()

def use_resource(n):
    with sem:
        with lock:
            active.append(n)
            print(f"Thread {n} active. Currently running: {sorted(active)}")
        time.sleep(0.1)
        with lock:
            active.remove(n)

threads = [threading.Thread(target=use_resource, args=(i,)) for i in range(6)]
for t in threads: t.start()
for t in threads: t.join()
print("All done")

### 9. Thread-Safe Singleton with Double-Checked Locking

Build a thread-safe Singleton class using double-checked locking to avoid lock overhead on every access after first creation.

In [ ]:
import threading

class Singleton:
    _instance = None
    _lock = threading.Lock()

    @classmethod
    def get_instance(cls):
        if cls._instance is None:                # first check (no lock, fast path)
            with cls._lock:
                if cls._instance is None:          # second check (inside lock, safe)
                    cls._instance = cls()
        return cls._instance

results = [None] * 10
def get_and_store(i):
    results[i] = Singleton.get_instance()

threads = [threading.Thread(target=get_and_store, args=(i,)) for i in range(10)]
for t in threads: t.start()
for t in threads: t.join()
print("All threads got the SAME instance:", len(set(id(r) for r in results)) == 1)

### 10. ThreadPoolExecutor.map for Simple Parallel I/O

Use `ThreadPoolExecutor.map()` to fetch 8 'URLs' (simulated I/O) concurrently with 4 workers, comparing against sequential timing.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def fetch(url):
    time.sleep(0.05)
    return f"content of {url}"

urls = [f"url-{i}" for i in range(8)]

start = time.perf_counter()
sequential = [fetch(u) for u in urls]
print(f"Sequential: {time.perf_counter()-start:.2f}s")

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    parallel = list(ex.map(fetch, urls))
print(f"Threaded (4 workers): {time.perf_counter()-start:.2f}s")
print("Same results:", sequential == parallel)

### 11. submit() + Future.result() for Individual Task Tracking

Use `executor.submit()` to launch tasks individually and collect results via `Future.result()`, demonstrating finer control than `.map()`.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

def process(item, multiplier):
    time.sleep(0.02)
    return item * multiplier

with ThreadPoolExecutor(max_workers=3) as ex:
    future_a = ex.submit(process, 5, 2)
    future_b = ex.submit(process, 10, 3)
    future_c = ex.submit(process, 7, 4)

    print("Result A:", future_a.result())
    print("Result B:", future_b.result())
    print("Result C:", future_c.result())

### 12. as_completed for Processing as Tasks Finish

Submit tasks with VARYING durations and use `as_completed()` to process each one the moment it's done, not in submission order.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def task(name, delay):
    time.sleep(delay)
    return name

with ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(task, name, delay): name for name, delay in
               [("slow", 0.2), ("fast", 0.02), ("medium", 0.1)]}
    for future in as_completed(futures):
        print(f"Completed: {future.result()}")

### 13. Handling Exceptions from Pooled Tasks

Submit a mix of succeeding/failing tasks to a `ThreadPoolExecutor` and properly catch exceptions via `Future.result()` per-task (not crashing the whole batch).

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def risky(n):
    if n == 3:
        raise ValueError(f"n={n} is unlucky")
    return n * 10

with ThreadPoolExecutor(max_workers=4) as ex:
    futures = [ex.submit(risky, i) for i in range(5)]
    for i, f in enumerate(futures):
        try:
            print(f"Task {i}: {f.result()}")
        except ValueError as e:
            print(f"Task {i}: FAILED - {e}")

### 14. Basic multiprocessing.Process

Create and run 2 separate processes (not threads), each computing something independently, demonstrating real OS-level parallelism setup.

In [ ]:
import multiprocessing as mp

def compute(n, name):
    result = sum(i*i for i in range(n))
    print(f"{name}: computed {result}")

if __name__ == "__main__":
    p1 = mp.Process(target=compute, args=(1_000_000, "Process-1"))
    p2 = mp.Process(target=compute, args=(1_000_000, "Process-2"))
    p1.start(); p2.start()
    p1.join(); p2.join()
    print("Both processes completed")

### 15. Measuring Real Speedup with multiprocessing.Pool

Compare sequential CPU-bound computation vs `Pool.map()` across 4 processes, measuring actual wall-clock speedup on this machine.

In [ ]:
import multiprocessing as mp
import time

def cpu_task(n):
    return sum(i*i for i in range(n))

if __name__ == "__main__":
    inputs = [3_000_000] * 4

    start = time.perf_counter()
    seq_results = [cpu_task(n) for n in inputs]
    seq_time = time.perf_counter() - start
    print(f"Sequential: {seq_time:.2f}s")

    start = time.perf_counter()
    with mp.Pool(processes=4) as pool:
        par_results = pool.map(cpu_task, inputs)
    par_time = time.perf_counter() - start
    print(f"Multiprocessing (4 procs): {par_time:.2f}s")
    print(f"Speedup: {seq_time/par_time:.2f}x")
    print("Results match:", seq_results == par_results)

### 16. Pool.apply_async for Non-Blocking Submission

Use `Pool.apply_async()` to submit tasks without blocking immediately, collecting results later via `.get()`.

In [ ]:
import multiprocessing as mp

def square(n):
    return n * n

if __name__ == "__main__":
    with mp.Pool(processes=3) as pool:
        async_results = [pool.apply_async(square, (i,)) for i in range(6)]
        final = [r.get(timeout=5) for r in async_results]   # blocks here, not at submission
    print(final)

### 17. Pool.imap for Lazy, Ordered Streaming Results

Use `Pool.imap()` (lazy, memory-efficient version of map) to process a large input lazily, printing results as they stream in.

In [ ]:
import multiprocessing as mp
import time

def slow_double(n):
    time.sleep(0.05)
    return n * 2

if __name__ == "__main__":
    with mp.Pool(processes=3) as pool:
        for result in pool.imap(slow_double, range(6)):
            print(f"Streamed result: {result}")    # arrives incrementally, in ORDER

### 18. ProcessPoolExecutor — the Modern High-Level API

Use `ProcessPoolExecutor` (mirrors `ThreadPoolExecutor`'s API) for CPU-bound work, showing the API symmetry between threading and multiprocessing executors.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def heavy(n):
    return sum(i*i for i in range(n))

if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=4) as ex:
        results = list(ex.map(heavy, [1_000_000]*4))
    print(f"Got {len(results)} results, all equal: {len(set(results)) == 1}")

### 19. Inter-Process Communication with multiprocessing.Queue

Build a producer process and consumer process communicating through a real `multiprocessing.Queue` (separate memory spaces, explicit message passing).

In [ ]:
import multiprocessing as mp

def producer(queue):
    for i in range(5):
        queue.put(f"message-{i}")
    queue.put(None)   # sentinel

def consumer(queue, output_list):
    while True:
        item = queue.get()
        if item is None:
            break
        output_list.append(item)

if __name__ == "__main__":
    queue = mp.Queue()
    manager = mp.Manager()
    output = manager.list()

    p1 = mp.Process(target=producer, args=(queue,))
    p2 = mp.Process(target=consumer, args=(queue, output))
    p1.start(); p2.start()
    p1.join(); p2.join()
    print(list(output))

### 20. Shared State with multiprocessing.Value and get_lock()

Use `mp.Value` (a shared primitive across processes) with its built-in lock to safely increment a counter from multiple processes.

In [ ]:
import multiprocessing as mp

def increment(shared_counter, times):
    for _ in range(times):
        with shared_counter.get_lock():
            shared_counter.value += 1

if __name__ == "__main__":
    counter = mp.Value("i", 0)    # 'i' = signed int
    processes = [mp.Process(target=increment, args=(counter, 1000)) for _ in range(4)]
    for p in processes: p.start()
    for p in processes: p.join()
    print(f"Final counter (expected 4000): {counter.value}")

### 21. Manager.dict() for Shared Mutable State Across Processes

Use `multiprocessing.Manager().dict()` to share a mutable dictionary across multiple processes, each writing their own key.

In [ ]:
import multiprocessing as mp

def worker(shared_dict, key, value):
    shared_dict[key] = value * value

if __name__ == "__main__":
    manager = mp.Manager()
    shared = manager.dict()

    processes = [mp.Process(target=worker, args=(shared, f"key{i}", i)) for i in range(5)]
    for p in processes: p.start()
    for p in processes: p.join()

    print(dict(shared))

### 22. Choosing Threads vs Processes — Empirical Proof

Run the SAME CPU-bound function with threads vs processes vs sequential, side by side, to empirically demonstrate WHY the choice matters (not just in theory).

In [ ]:
import threading, multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import time

def cpu_bound(n):
    return sum(i*i for i in range(n))

if __name__ == "__main__":
    N = 2_000_000
    tasks = [N]*4

    start = time.perf_counter()
    [cpu_bound(n) for n in tasks]
    t_seq = time.perf_counter() - start

    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=4) as ex:
        list(ex.map(cpu_bound, tasks))
    t_thread = time.perf_counter() - start

    start = time.perf_counter()
    with ProcessPoolExecutor(max_workers=4) as ex:
        list(ex.map(cpu_bound, tasks))
    t_process = time.perf_counter() - start

    print(f"Sequential:       {t_seq:.2f}s")
    print(f"Threaded (GIL):   {t_thread:.2f}s  <- often NOT faster, sometimes slower!")
    print(f"Multiprocessing:  {t_process:.2f}s  <- actually parallel, usually faster on multi-core")

### 23. Hybrid Pipeline: I/O Threads Feed CPU Processes

Build a 2-stage pipeline: Stage 1 uses THREADS for simulated I/O fetch, Stage 2 uses PROCESSES for CPU-bound transform — matching tool to workload per stage.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import time

def fetch_io(item_id):
    time.sleep(0.02)             # I/O-bound
    return {"id": item_id, "raw": item_id * 7}

def transform_cpu(record):
    total = sum(i for i in range(record["raw"] * 1000))    # CPU-bound
    return {**record, "computed": total}

if __name__ == "__main__":
    ids = list(range(8))

    with ThreadPoolExecutor(max_workers=4) as io_ex:
        fetched = list(io_ex.map(fetch_io, ids))

    with ProcessPoolExecutor(max_workers=4) as cpu_ex:
        transformed = list(cpu_ex.map(transform_cpu, fetched))

    print(f"Pipeline processed {len(transformed)} records")
    print("Sample:", transformed[0])

### 24. Deadlock Demonstration and Fix (Lock Ordering)

Demonstrate (safely, with a timeout to avoid hanging forever) how acquiring 2 locks in INCONSISTENT order across threads can deadlock, and fix it with consistent ordering.

In [ ]:
import threading

lock_a = threading.Lock()
lock_b = threading.Lock()

def safe_worker_1():
    # Always acquire in the SAME global order: A then B
    with lock_a:
        with lock_b:
            pass   # do work
    return "worker1 done"

def safe_worker_2():
    with lock_a:        # same order as worker_1 - prevents the classic deadlock pattern
        with lock_b:
            pass
    return "worker2 done"

results = []
def run(fn):
    results.append(fn())

t1 = threading.Thread(target=run, args=(safe_worker_1,))
t2 = threading.Thread(target=run, args=(safe_worker_2,))
t1.start(); t2.start()
t1.join(timeout=2); t2.join(timeout=2)
print("Both completed without deadlock:", results)
print("Lesson: ALWAYS acquire multiple locks in a single, consistent global order")

### 25. Graceful Worker Pool Shutdown with Exception Isolation

Build a production-style pattern: a `ThreadPoolExecutor` processing a batch where SOME tasks fail, collecting both successes and failures separately without crashing the batch.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def process_record(record):
    if record["id"] % 4 == 0:
        raise ValueError(f"Record {record['id']} failed validation")
    return {"id": record["id"], "status": "processed"}

records = [{"id": i} for i in range(12)]
successes, failures = [], []

with ThreadPoolExecutor(max_workers=4) as ex:
    futures = {ex.submit(process_record, r): r for r in records}
    for future in as_completed(futures):
        record = futures[future]
        try:
            successes.append(future.result())
        except ValueError as e:
            failures.append({"id": record["id"], "error": str(e)})

print(f"Succeeded: {len(successes)}, Failed: {len(failures)}")
print("Failures:", failures)